# IBM Applied Data Science Capstone: SpaceX Falcon 9 First Stage Landing Prediction
## Lab 4: Exploratory Data Analysis with SQL
**Author:** Shravani D  
**GitHub Repository:** [SpaceX-Falcon9-Landing-Prediction](https://github.com/ShravaniD/SpaceX-Falcon9-Landing-Prediction)

---
### Objective
Execute structured SQL queries on `spacex.csv` loaded into an in-memory SQLite database (`SPACEXTBL`) to answer key business, operational, and historical questions regarding launch sites, payload capacities, booster versions, and landing outcomes.


In [1]:
import sqlite3
import pandas as pd
import os

csv_path = os.path.join("..", "data", "spacex.csv")
df_sql = pd.read_csv(csv_path)

# Sanitize column names for SQLite
df_sql.columns = [c.replace(' ', '_').replace('(', '').replace(')', '').replace('/', '_') for c in df_sql.columns]

conn = sqlite3.connect(':memory:')
df_sql.to_sql('SPACEXTBL', conn, if_exists='replace', index=False)
print("Connected to SQLite and loaded SPACEXTBL table with", len(df_sql), "records.")


### Query 1: Names of Unique Launch Sites


In [1]:
pd.read_sql("SELECT DISTINCT Launch_Site FROM SPACEXTBL", conn)


### Query 2: 5 Records Where Launch Sites Begin with 'CCA'


In [1]:
pd.read_sql("SELECT * FROM SPACEXTBL WHERE Launch_Site LIKE 'CCA%' LIMIT 5", conn)


### Query 3: Total Payload Mass Carried by Boosters Launched by NASA (CRS)


In [1]:
pd.read_sql("SELECT SUM(PAYLOAD_MASS__KG_) AS Total_Payload_Mass_NASA_CRS FROM SPACEXTBL WHERE Customer = 'NASA (CRS)'", conn)


### Query 4: Average Payload Mass Carried by Booster Version F9 v1.1


In [1]:
pd.read_sql("SELECT AVG(PAYLOAD_MASS__KG_) AS Avg_Payload_Mass_F9_v1_1 FROM SPACEXTBL WHERE Booster_Version LIKE 'F9 v1.1%'", conn)


### Query 5: Date When First Successful Ground Pad Landing Was Achieved


In [1]:
pd.read_sql("SELECT MIN(Date) AS First_Successful_Ground_Landing_Date FROM SPACEXTBL WHERE Landing_Outcome = 'Success (ground pad)'", conn)


### Query 6: Boosters with Success in Drone Ship and Payload Mass Between 4,000 and 6,000 kg


In [1]:
pd.read_sql("SELECT Booster_Version, PAYLOAD_MASS__KG_, Landing_Outcome FROM SPACEXTBL WHERE Landing_Outcome = 'Success (drone ship)' AND PAYLOAD_MASS__KG_ BETWEEN 4000 AND 6000", conn)


### Query 7: Total Number of Successful and Failure Mission Outcomes


In [1]:
pd.read_sql("SELECT Mission_Outcome, COUNT(*) AS Total_Count FROM SPACEXTBL GROUP BY Mission_Outcome", conn)


### Query 8: Booster Versions That Have Carried the Maximum Payload Mass (Subquery)


In [1]:
pd.read_sql("SELECT Booster_Version, PAYLOAD_MASS__KG_ FROM SPACEXTBL WHERE PAYLOAD_MASS__KG_ = (SELECT MAX(PAYLOAD_MASS__KG_) FROM SPACEXTBL)", conn)


### Query 9: Failed Drone Ship Landings in 2015 by Month, Booster Version, and Launch Site


In [1]:
pd.read_sql("SELECT substr(Date, 6, 2) AS Month, Booster_Version, Launch_Site, Landing_Outcome FROM SPACEXTBL WHERE Landing_Outcome = 'Failure (drone ship)' AND Date LIKE '2015%'", conn)


### Query 10: Rank Landing Outcomes Between 2010-06-04 and 2017-03-20 in Descending Order


In [1]:
pd.read_sql("SELECT Landing_Outcome, COUNT(*) AS Outcome_Count FROM SPACEXTBL WHERE Date BETWEEN '2010-06-04' AND '2017-03-20' GROUP BY Landing_Outcome ORDER BY Outcome_Count DESC", conn)


### Summary of SQL Findings
- NASA (CRS) is the largest institutional customer by total payload mass (45,596 kg).
- The maximum payload carried is 15,600 kg, achieved by heavy Starlink deployment missions using Block 5 boosters.
- First ground pad landing took place on December 22, 2015 at Cape Canaveral LZ-1.
- Drone ship landings achieved operational maturity post-2016 after early experimental landing tests in 2015.
